# Ejercicios: máquinas de Turing básicas

En este cuaderno diseñarás y probarás máquinas de Turing para problemas concretos.

Cada ejercicio pide completar las transiciones de una MT y verificar su funcionamiento con casos de prueba.

**Artículos asociados:** [Máquinas de Turing](../../03-computabilidad/04-maquinas-de-turing.md), [Decidibilidad y reconocibilidad](../../03-computabilidad/02-decidibilidad-y-reconocibilidad.md)

In [ ]:
# Simulador de MT (igual que en el cuaderno de ejemplos)
class TuringMachine:
    BLANK = '_'

    def __init__(self, states, initial, accepting, rejecting, transitions):
        self.states = set(states)
        self.initial = initial
        self.accepting = accepting
        self.rejecting = rejecting
        self.transitions = transitions

    def run(self, input_string, max_steps=2000):
        tape = list(input_string) if input_string else [self.BLANK]
        head = 0
        state = self.initial
        steps = 0
        while steps < max_steps:
            if head >= len(tape):
                tape.append(self.BLANK)
            if head < 0:
                tape.insert(0, self.BLANK)
                head = 0
            symbol = tape[head]
            if state == self.accepting:
                return True
            if state == self.rejecting:
                return False
            key = (state, symbol)
            if key not in self.transitions:
                return False
            new_state, write, direction = self.transitions[key]
            tape[head] = write
            state = new_state
            head += 1 if direction == 'R' else -1
            steps += 1
        raise RuntimeError('Límite de pasos alcanzado')

def verificar(mt, casos):
    """Verifica una lista de casos (entrada, resultado_esperado)."""
    todos_ok = True
    for entrada, esperado in casos:
        resultado = mt.run(entrada)
        ok = '✓' if resultado == esperado else '✗'
        print(f'  {ok} entrada={repr(entrada):12} esperado={esperado} obtenido={resultado}')
        if resultado != esperado:
            todos_ok = False
    print('Todos los casos correctos.' if todos_ok else 'Hay errores.')

print('Simulador y utilidades cargados.')

## Ejercicio 1: reconocer cadenas con igual número de `a` y `b`

Diseña una MT que acepte el lenguaje $\{a^n b^n : n \geq 0\}$.

**Estrategia sugerida:** marcar un par (una `a` y una `b`) a la vez hasta que no queden más. Si al terminar quedan solo marcas, aceptar.

Completa el diccionario `transitions_anbn`.

In [ ]:
# Completa las transiciones. Convenciones:
# - 'X' marca una 'a' ya emparejada
# - 'Y' marca una 'b' ya emparejada
# Estados sugeridos: q0 (buscar a), q1 (buscar b), q2 (volver), qa, qr

transitions_anbn = {
    # TODO: completar
    # Pista:
    #   q0: leer 'a' → marcarla como 'X' y pasar a q1
    #   q0: leer 'X' o 'Y' → avanzar (ya emparejados)
    #   q0: leer '_' → aceptar (llegamos al final con todo emparejado)
    #   q1: avanzar hasta la primera 'b'
    #   q1: leer 'b' → marcarla como 'Y' y pasar a q2
    #   q1: leer '_' → rechazar (no hay b para la a)
    #   q2: retroceder hasta el inicio
    #   q2: leer '_' → reiniciar en q0
}

# Descomenta y ejecuta cuando hayas completado las transiciones:
# mt_anbn = TuringMachine(
#     states={'q0','q1','q2','qa','qr'},
#     initial='q0', accepting='qa', rejecting='qr',
#     transitions=transitions_anbn
# )
# casos = [
#     ('', True), ('ab', True), ('aabb', True), ('aaabbb', True),
#     ('a', False), ('b', False), ('ba', False), ('aab', False), ('abab', False)
# ]
# verificar(mt_anbn, casos)

### Solución de referencia para el ejercicio 1

In [ ]:
transitions_anbn = {
    ('q0', 'a'): ('q1', 'X', 'R'),
    ('q0', 'X'): ('q0', 'X', 'R'),
    ('q0', 'Y'): ('q0', 'Y', 'R'),
    ('q0', '_'): ('qa', '_', 'R'),  # todo emparejado
    ('q0', 'b'): ('qr', 'b', 'R'),  # b antes de terminar las a: rechazar

    ('q1', 'a'): ('q1', 'a', 'R'),  # avanzar por las a no marcadas
    ('q1', 'X'): ('q1', 'X', 'R'),  # avanzar por las a ya marcadas
    ('q1', 'Y'): ('q1', 'Y', 'R'),  # avanzar por las b ya marcadas
    ('q1', 'b'): ('q2', 'Y', 'L'),  # marcar la primera b y retroceder
    ('q1', '_'): ('qr', '_', 'L'),  # no hay b: rechazar

    ('q2', 'a'): ('q2', 'a', 'L'),  # retroceder
    ('q2', 'X'): ('q2', 'X', 'L'),
    ('q2', 'Y'): ('q2', 'Y', 'L'),
    ('q2', '_'): ('q0', '_', 'R'),  # llegamos al inicio: reiniciar
}

mt_anbn = TuringMachine(
    states={'q0','q1','q2','qa','qr'},
    initial='q0', accepting='qa', rejecting='qr',
    transitions=transitions_anbn
)

casos = [
    ('',      True),
    ('ab',    True),
    ('aabb',  True),
    ('aaabbb',True),
    ('a',     False),
    ('b',     False),
    ('ba',    False),
    ('aab',   False),
    ('abab',  False),
]
print('Verificación de a^n b^n:')
verificar(mt_anbn, casos)

## Ejercicio 2: comprobar si una cadena es de la forma $0^*1^*$

Diseña una MT que acepte todas las cadenas sobre $\{0,1\}$ donde todos los `0` preceden a todos los `1` (incluyendo la cadena vacía, cadenas solo con `0` y cadenas solo con `1`).

Esta es una tarea simple: se puede hacer con 3 estados. El truco es detectar el patrón `10` que indicaría que un `1` va antes que un `0`.

In [ ]:
# Completa las transiciones para reconocer 0*1*
# Estados sugeridos: q_ceros (leyendo ceros), q_unos (leyendo unos), qa, qr

transitions_0s1s = {
    # TODO: completar
    # Pista: empezar en q_ceros
    # q_ceros: leer '0' → seguir en q_ceros; leer '1' → pasar a q_unos
    # q_ceros: leer '_' → aceptar
    # q_unos: leer '1' → seguir en q_unos
    # q_unos: leer '0' → rechazar (un 0 después de un 1)
    # q_unos: leer '_' → aceptar
}

# Descomenta cuando hayas completado:
# mt_0s1s = TuringMachine(
#     states={'q_ceros','q_unos','qa','qr'},
#     initial='q_ceros', accepting='qa', rejecting='qr',
#     transitions=transitions_0s1s
# )
# casos = [
#     ('', True), ('0', True), ('1', True), ('01', True),
#     ('0011', True), ('0001111', True),
#     ('10', False), ('010', False), ('101', False)
# ]
# verificar(mt_0s1s, casos)

In [ ]:
# Solución de referencia
transitions_0s1s = {
    ('q_ceros', '0'): ('q_ceros', '0', 'R'),
    ('q_ceros', '1'): ('q_unos',  '1', 'R'),
    ('q_ceros', '_'): ('qa',      '_', 'R'),
    ('q_unos',  '1'): ('q_unos',  '1', 'R'),
    ('q_unos',  '0'): ('qr',      '0', 'R'),
    ('q_unos',  '_'): ('qa',      '_', 'R'),
}

mt_0s1s = TuringMachine(
    states={'q_ceros','q_unos','qa','qr'},
    initial='q_ceros', accepting='qa', rejecting='qr',
    transitions=transitions_0s1s
)

casos = [
    ('',       True),
    ('0',      True),
    ('1',      True),
    ('01',     True),
    ('0011',   True),
    ('0001111',True),
    ('10',     False),
    ('010',    False),
    ('101',    False),
]
print('Verificación de 0*1*:')
verificar(mt_0s1s, casos)

## Ejercicio 3: número de pasos y complejidad temporal

Mide experimentalmente cuántos pasos necesitan las dos MT anteriores para cadenas de longitud creciente. ¿Son lineales o cuadráticas?

In [ ]:
def contar_pasos(mt, entrada, max_steps=10000):
    tape = list(entrada) if entrada else [mt.BLANK]
    head = 0
    state = mt.initial
    steps = 0
    while steps < max_steps:
        if head >= len(tape):
            tape.append(mt.BLANK)
        if head < 0:
            tape.insert(0, mt.BLANK)
            head = 0
        symbol = tape[head]
        if state in (mt.accepting, mt.rejecting):
            return steps
        key = (state, symbol)
        if key not in mt.transitions:
            return steps
        new_state, write, direction = mt.transitions[key]
        tape[head] = write
        state = new_state
        head += 1 if direction == 'R' else -1
        steps += 1
    return steps

print('Pasos de la MT a^n b^n (cadenas a^n b^n):')
print(f'{"n":>5}  {"longitud":>8}  {"pasos":>8}  {"pasos/n^2":>10}')
for n in range(1, 9):
    entrada = 'a'*n + 'b'*n
    p = contar_pasos(mt_anbn, entrada)
    print(f'{n:>5}  {len(entrada):>8}  {p:>8}  {p/(len(entrada)**2):>10.4f}')

print()
print('Pasos de la MT 0*1* (cadenas 0^n 1^n):')
print(f'{"n":>5}  {"longitud":>8}  {"pasos":>8}  {"pasos/n":>10}')
for n in range(1, 9):
    entrada = '0'*n + '1'*n
    p = contar_pasos(mt_0s1s, entrada)
    print(f'{n:>5}  {len(entrada):>8}  {p:>8}  {p/len(entrada):>10.4f}')

## Reflexión

- La MT para $0^*1^*$ es **lineal**: solo recorre la cinta una vez.
- La MT para $a^n b^n$ es **cuadrática**: en cada iteración recorre casi toda la cinta para encontrar el siguiente par.

Ambas son decidibles. La diferencia está en la eficiencia, no en la computabilidad.

**Para reflexionar:** ¿puedes diseñar una MT para $a^n b^n$ que solo necesite tiempo lineal? (Pista: requiere una MT de 2 cintas o una estrategia diferente de marcado.)

## Soluciones de referencia

Descomenta las celdas para ver las soluciones.

In [ ]:
# --- Solución de referencia: Ejercicio 1 ---
# MT que acepta {0^n 1^n : n >= 1}:
# 1. Marcar el primer 0 no marcado como 'X'.
# 2. Mover a la derecha hasta encontrar el primer 1 no marcado.
# 3. Marcarlo como 'Y'.
# 4. Volver a la izquierda al primer X, luego al siguiente 0.
# 5. Repetir hasta no quedar 0s. Verificar que tampoco queden 1s.
# Número de pasos: O(n^2) — cada iteración recorre hasta n celdas.

# --- Solución de referencia: Ejercicio 2 ---
# MT que duplica: para entrada w, produce ww.
# Estrategia: copiar símbolo a símbolo al final de la cinta.
# Complejidad: O(|w|^2) — para cada símbolo, desplazarse hasta el final.

# --- Solución de referencia: Ejercicio 3 ---
# contar_pasos para la MT de 0^*1^* en '000111': ~3*7 = 21 pasos.
# La MT de paridad es lineal: recorre la cinta una vez -> O(n) pasos.
# La MT de 0^n1^n es cuadrática: cada par requiere recorrer toda la cinta -> O(n^2) pasos.
print('Referencia lista: descomenta el código de tu ejercicio para comparar.')

In [ ]:
# ── Verificación automática ──────────────────────────────────────────────────

def simular_mt_anbn(entrada):
    """MT que acepta {a^n b^n | n>=1} por marcado."""
    cinta = list(entrada) + ['_']
    cabezal = 0
    estado = 'q0'
    for _ in range(10000):
        simbolo = cinta[cabezal] if cabezal < len(cinta) else '_'
        if estado == 'q0':
            if simbolo == 'a':
                cinta[cabezal] = 'X'; estado = 'q1'; cabezal += 1
            elif simbolo == 'Y':
                estado = 'q3'; cabezal += 1
            else: return False
        elif estado == 'q1':
            if simbolo in ('a', 'Y'): cabezal += 1
            elif simbolo == 'b': cinta[cabezal] = 'Y'; estado = 'q2'; cabezal -= 1
            else: return False
        elif estado == 'q2':
            if simbolo in ('a', 'Y'): cabezal -= 1
            elif simbolo == 'X': estado = 'q0'; cabezal += 1
            else: return False
        elif estado == 'q3':
            if simbolo == 'Y': cabezal += 1
            elif simbolo == '_': return True
            else: return False
    return False

assert simular_mt_anbn("ab"), "ab debe aceptarse"
assert simular_mt_anbn("aabb"), "aabb debe aceptarse"
assert simular_mt_anbn("aaabbb"), "aaabbb debe aceptarse"
assert not simular_mt_anbn("aab"), "aab no debe aceptarse"
assert not simular_mt_anbn("abb"), "abb no debe aceptarse"
assert not simular_mt_anbn("ba"), "ba no debe aceptarse"
print("✓ MT para a^n b^n correcta")